In [40]:
import numpy as np
import pandas as pd

In [41]:
df=pd.read_csv("finalDataSet.csv")

In [42]:
df.head()

,Date,Adj Close,Close,High,Low,Open,Volume,SYMBOL,TARGET_CLOSE
0,2024-01-01,681.303101,702.849976,714.400024,693.000000,709.849976,175009,360ONE,676.900024
1,2024-01-02,656.148621,676.900024,699.950012,675.000000,695.250000,348865,360ONE,655.250000
2,2024-01-03,635.162415,655.250000,682.049988,652.900024,681.099976,203415,360ONE,653.849976
3,2024-01-04,633.805237,653.849976,662.049988,651.250000,660.000000,133259,360ONE,662.450012
4,2024-01-05,642.141602,662.450012,664.950012,644.049988,653.849976,1231143,360ONE,672.450012


In [43]:
df.shape

(289189, 9)

In [44]:
# ============================================================
# BASIC DATA PREPARATION
# ============================================================

df["Date"] = pd.to_datetime(df["Date"])

df = (
    df
    .sort_values(["SYMBOL", "Date"])
    .reset_index(drop=True)
)

print("DATASET READY")
print("----------------")
print("Shape:", df.shape)
print("Stocks:", df["SYMBOL"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

DATASET READY
----------------
Shape: (289189, 9)
Stocks: 502
Date range: 2024-01-01 00:00:00 to 2026-06-18 00:00:00

Columns:
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'SYMBOL', 'TARGET_CLOSE']

Missing values:
0


In [45]:
# ============================================================
# FUTURE RETURN TARGETS
# ============================================================

# Sort stock-wise so shift() never crosses between stocks
df = df.sort_values(["SYMBOL", "Date"]).reset_index(drop=True)

# Tomorrow's return
df["TARGET_RETURN_1D"] = (
    df.groupby("SYMBOL")["Close"].shift(-1) / df["Close"]
) - 1

# 3-day forward return
df["TARGET_RETURN_3D"] = (
    df.groupby("SYMBOL")["Close"].shift(-3) / df["Close"]
) - 1

# 5-day forward return
df["TARGET_RETURN_5D"] = (
    df.groupby("SYMBOL")["Close"].shift(-5) / df["Close"]
) - 1

print("FUTURE RETURN TARGETS")
print("---------------------")
print("1D mean :", df["TARGET_RETURN_1D"].mean())
print("1D std  :", df["TARGET_RETURN_1D"].std())

print("3D mean :", df["TARGET_RETURN_3D"].mean())
print("3D std  :", df["TARGET_RETURN_3D"].std())

print("5D mean :", df["TARGET_RETURN_5D"].mean())
print("5D std  :", df["TARGET_RETURN_5D"].std())

print("\nMissing values:")
print(df[
    ["TARGET_RETURN_1D", "TARGET_RETURN_3D", "TARGET_RETURN_5D"]
].isnull().sum())

FUTURE RETURN TARGETS
---------------------
1D mean : 0.0006729577665829173
1D std  : 0.02413626657496684
3D mean : 0.0019911151672122418
3D std  : 0.04208515801991511
5D mean : 0.0032055938117117526
5D std  : 0.054351198739957626

Missing values:
TARGET_RETURN_1D     502
TARGET_RETURN_3D    1506
TARGET_RETURN_5D    2508
dtype: int64


In [46]:
# ============================================================
# TECHNICAL + MARKET FEATURES
# ============================================================

g = df.groupby("SYMBOL", group_keys=False)

# -----------------------------
# RETURNS / MOMENTUM
# -----------------------------

df["RET_1D"] = g["Close"].pct_change(1)
df["RET_3D"] = g["Close"].pct_change(3)
df["RET_5D"] = g["Close"].pct_change(5)
df["RET_10D"] = g["Close"].pct_change(10)
df["RET_20D"] = g["Close"].pct_change(20)

# -----------------------------
# PRICE LAGS
# -----------------------------

for lag in [1, 2, 3, 5, 10]:
    df[f"CLOSE_LAG_{lag}"] = g["Close"].shift(lag)

# -----------------------------
# MOVING AVERAGES
# -----------------------------

for window in [5, 10, 20, 50]:
    df[f"SMA_{window}"] = g["Close"].transform(
        lambda x: x.rolling(window).mean()
    )

for window in [5, 10, 20]:
    df[f"EMA_{window}"] = g["Close"].transform(
        lambda x: x.ewm(span=window, adjust=False).mean()
    )

# Price relative to moving averages
df["PRICE_SMA20"] = df["Close"] / df["SMA_20"] - 1
df["PRICE_SMA50"] = df["Close"] / df["SMA_50"] - 1

# -----------------------------
# VOLATILITY
# -----------------------------

for window in [5, 10, 20]:
    df[f"VOLATILITY_{window}"] = g["RET_1D"].transform(
        lambda x: x.rolling(window).std()
    )

# -----------------------------
# RSI 14
# -----------------------------

delta = g["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(df["SYMBOL"]).transform(
    lambda x: x.rolling(14).mean()
)

avg_loss = loss.groupby(df["SYMBOL"]).transform(
    lambda x: x.rolling(14).mean()
)

rs = avg_gain / avg_loss

df["RSI_14"] = 100 - (100 / (1 + rs))

# -----------------------------
# MACD
# -----------------------------

df["EMA_12"] = g["Close"].transform(
    lambda x: x.ewm(span=12, adjust=False).mean()
)

df["EMA_26"] = g["Close"].transform(
    lambda x: x.ewm(span=26, adjust=False).mean()
)

df["MACD"] = df["EMA_12"] - df["EMA_26"]

df["MACD_SIGNAL"] = (
    df.groupby("SYMBOL")["MACD"]
    .transform(lambda x: x.ewm(span=9, adjust=False).mean())
)

df["MACD_HIST"] = df["MACD"] - df["MACD_SIGNAL"]

# -----------------------------
# BOLLINGER BANDS
# -----------------------------

df["BB_MIDDLE"] = df["SMA_20"]

df["BB_STD"] = g["Close"].transform(
    lambda x: x.rolling(20).std()
)

df["BB_UPPER"] = df["BB_MIDDLE"] + 2 * df["BB_STD"]
df["BB_LOWER"] = df["BB_MIDDLE"] - 2 * df["BB_STD"]

df["BB_WIDTH"] = (
    (df["BB_UPPER"] - df["BB_LOWER"])
    / df["BB_MIDDLE"]
)

df["BB_POSITION"] = (
    (df["Close"] - df["BB_LOWER"])
    / (df["BB_UPPER"] - df["BB_LOWER"])
)

# -----------------------------
# ATR 14
# -----------------------------

prev_close = g["Close"].shift(1)

tr1 = df["High"] - df["Low"]
tr2 = (df["High"] - prev_close).abs()
tr3 = (df["Low"] - prev_close).abs()

df["TRUE_RANGE"] = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

df["ATR_14"] = (
    df.groupby("SYMBOL")["TRUE_RANGE"]
    .transform(lambda x: x.rolling(14).mean())
)

df["ATR_PERCENT"] = df["ATR_14"] / df["Close"]

# -----------------------------
# STOCHASTIC OSCILLATOR
# -----------------------------

low_14 = g["Low"].transform(
    lambda x: x.rolling(14).min()
)

high_14 = g["High"].transform(
    lambda x: x.rolling(14).max()
)

df["STOCH_K"] = (
    (df["Close"] - low_14)
    / (high_14 - low_14)
) * 100

df["STOCH_D"] = (
    df.groupby("SYMBOL")["STOCH_K"]
    .transform(lambda x: x.rolling(3).mean())
)

# -----------------------------
# VOLUME FEATURES
# -----------------------------

df["VOLUME_LAG_1"] = g["Volume"].shift(1)

df["VOLUME_SMA20"] = (
    g["Volume"]
    .transform(lambda x: x.rolling(20).mean())
)

df["VOLUME_RATIO"] = (
    df["Volume"] / df["VOLUME_SMA20"]
)

df["VOLUME_RETURN_CORR"] = (
    df.groupby("SYMBOL")
    .apply(
        lambda x: x["Volume"]
        .rolling(20)
        .corr(x["RET_1D"])
    )
    .reset_index(level=0, drop=True)
)

# -----------------------------
# CLEAN FEATURE DATA
# -----------------------------

df = df.replace([np.inf, -np.inf], np.nan)

print("FEATURE ENGINEERING COMPLETE")
print("-----------------------------")
print("Rows :", len(df))
print("Columns :", len(df.columns))

print("\nNew features:")
print([
    c for c in df.columns
    if c not in [
        "Date", "Adj Close", "Close", "High",
        "Low", "Open", "Volume", "SYMBOL",
        "TARGET_CLOSE",
        "TARGET_RETURN_1D",
        "TARGET_RETURN_3D",
        "TARGET_RETURN_5D"
    ]
])

FEATURE ENGINEERING COMPLETE
-----------------------------
Rows : 289189
Columns : 55

New features:
['RET_1D', 'RET_3D', 'RET_5D', 'RET_10D', 'RET_20D', 'CLOSE_LAG_1', 'CLOSE_LAG_2', 'CLOSE_LAG_3', 'CLOSE_LAG_5', 'CLOSE_LAG_10', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50', 'EMA_5', 'EMA_10', 'EMA_20', 'PRICE_SMA20', 'PRICE_SMA50', 'VOLATILITY_5', 'VOLATILITY_10', 'VOLATILITY_20', 'RSI_14', 'EMA_12', 'EMA_26', 'MACD', 'MACD_SIGNAL', 'MACD_HIST', 'BB_MIDDLE', 'BB_STD', 'BB_UPPER', 'BB_LOWER', 'BB_WIDTH', 'BB_POSITION', 'TRUE_RANGE', 'ATR_14', 'ATR_PERCENT', 'STOCH_K', 'STOCH_D', 'VOLUME_LAG_1', 'VOLUME_SMA20', 'VOLUME_RATIO', 'VOLUME_RETURN_CORR']


/var/folders/lv/g3cgrt6s4h59rbq5tjq1f2qr0000gn/T/ipykernel_16762/1875988408.py:177: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [47]:
# ============================================================
# CLEAN FEATURE DATA
# ============================================================

df = df.replace([np.inf, -np.inf], np.nan)

# Features we will use
feature_cols = [
    "Close",
    "High",
    "Low",
    "Open",
    "Volume",

    "RET_1D",
    "RET_3D",
    "RET_5D",
    "RET_10D",
    "RET_20D",

    "CLOSE_LAG_1",
    "CLOSE_LAG_2",
    "CLOSE_LAG_3",
    "CLOSE_LAG_5",
    "CLOSE_LAG_10",

    "SMA_5",
    "SMA_10",
    "SMA_20",
    "SMA_50",

    "EMA_5",
    "EMA_10",
    "EMA_20",

    "PRICE_SMA20",
    "PRICE_SMA50",

    "VOLATILITY_5",
    "VOLATILITY_10",
    "VOLATILITY_20",

    "RSI_14",

    "MACD",
    "MACD_SIGNAL",
    "MACD_HIST",

    "BB_MIDDLE",
    "BB_UPPER",
    "BB_LOWER",
    "BB_WIDTH",
    "BB_POSITION",

    "ATR_14",
    "ATR_PERCENT",

    "STOCH_K",
    "STOCH_D",

    "VOLUME_LAG_1",
    "VOLUME_SMA20",
    "VOLUME_RATIO",
    "VOLUME_RETURN_CORR"
]

# Keep only rows where features and 1D target exist
df_model = df.dropna(
    subset=feature_cols + ["TARGET_RETURN_1D"]
).copy()

df_model = df_model.sort_values(
    ["Date", "SYMBOL"]
).reset_index(drop=True)

print("CLEAN DATASET")
print("----------------")
print("Original rows :", len(df))
print("Model rows    :", len(df_model))
print("Features      :", len(feature_cols))
print("Missing values:", df_model[feature_cols + ["TARGET_RETURN_1D"]].isnull().sum().sum())
print("Stocks        :", df_model["SYMBOL"].nunique())

print("\nDate range:")
print(df_model["Date"].min(), "to", df_model["Date"].max())

print("\nSYMBOL included:", "SYMBOL" in feature_cols)

CLEAN DATASET
----------------
Original rows : 289189
Model rows    : 264181
Features      : 44
Missing values: 0
Stocks        : 500

Date range:
2024-03-13 00:00:00 to 2026-06-17 00:00:00

SYMBOL included: False


In [48]:
# ============================================================
# CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# Make sure data is correctly ordered
df_model = df_model.sort_values(
    ["Date", "SYMBOL"]
).reset_index(drop=True)

X_return = df_model[feature_cols].copy()
y_return = df_model["TARGET_RETURN_1D"].copy()

# ------------------------------------------------------------
# DATE-BASED SPLIT
# ------------------------------------------------------------

unique_dates = np.sort(df_model["Date"].unique())

train_end = unique_dates[int(len(unique_dates) * 0.70)]
validation_end = unique_dates[int(len(unique_dates) * 0.80)]

return_train_mask = df_model["Date"] < train_end
return_validation_mask = (
    (df_model["Date"] >= train_end) &
    (df_model["Date"] < validation_end)
)
return_test_mask = df_model["Date"] >= validation_end

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

X_return_train = X_return.loc[return_train_mask].copy()
y_return_train = y_return.loc[return_train_mask].copy()

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

X_return_validation = X_return.loc[return_validation_mask].copy()
y_return_validation = y_return.loc[return_validation_mask].copy()

# ------------------------------------------------------------
# FINAL TEST
# ------------------------------------------------------------

X_return_test = X_return.loc[return_test_mask].copy()
y_return_test = y_return.loc[return_test_mask].copy()

# ------------------------------------------------------------
# INFORMATION
# ------------------------------------------------------------

print("RETURN MODEL - CHRONOLOGICAL SPLIT")
print("----------------------------------")

print("Training rows   :", len(X_return_train))
print("Validation rows :", len(X_return_validation))
print("Testing rows    :", len(X_return_test))

print("Features        :", X_return_train.shape[1])
print("Stocks          :", df_model["SYMBOL"].nunique())

print("\nTraining period:")
print(
    df_model.loc[return_train_mask, "Date"].min(),
    "to",
    df_model.loc[return_train_mask, "Date"].max()
)

print("\nValidation period:")
print(
    df_model.loc[return_validation_mask, "Date"].min(),
    "to",
    df_model.loc[return_validation_mask, "Date"].max()
)

print("\nTesting period:")
print(
    df_model.loc[return_test_mask, "Date"].min(),
    "to",
    df_model.loc[return_test_mask, "Date"].max()
)

print("\nSYMBOL included:", "SYMBOL" in feature_cols)

RETURN MODEL - CHRONOLOGICAL SPLIT
----------------------------------
Training rows   : 181230
Validation rows : 27206
Testing rows    : 55745
Features        : 44
Stocks          : 500

Training period:
2024-03-13 00:00:00 to 2025-10-13 00:00:00

Validation period:
2025-10-14 00:00:00 to 2026-01-02 00:00:00

Testing period:
2026-01-05 00:00:00 to 2026-06-17 00:00:00

SYMBOL included: False


In [49]:
# ============================================================
# ZERO-RETURN BASELINE
# ============================================================

import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Predict exactly 0% return for every test observation
return_baseline_pred = np.zeros(len(y_return_test))

baseline_mae = mean_absolute_error(
    y_return_test,
    return_baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_return_test,
        return_baseline_pred
    )
)

baseline_r2 = r2_score(
    y_return_test,
    return_baseline_pred
)

print("ZERO-RETURN BASELINE")
print("--------------------")
print("R²   :", baseline_r2)
print("MAE  :", baseline_mae)
print("RMSE :", baseline_rmse)

ZERO-RETURN BASELINE
--------------------
R²   : -0.00018258823950234593
MAE  : 0.0174313790648264
RMSE : 0.02472059107163854


In [50]:
# ============================================================
# LIGHTGBM - 1D RETURN REGRESSION
# ============================================================

import lightgbm as lgb
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

lgb_return_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    objective="regression",

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

lgb_return_model.fit(
    X_return_train,
    y_return_train,

    eval_set=[
        (X_return_validation, y_return_validation)
    ],

    eval_metric="l1",

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=False
        ),
        lgb.log_evaluation(0)
    ]
)

# ------------------------------------------------------------
# VALIDATION PREDICTION
# ------------------------------------------------------------

lgb_return_validation_pred = (
    lgb_return_model.predict(
        X_return_validation
    )
)

# ------------------------------------------------------------
# REGRESSION METRICS
# ------------------------------------------------------------

lgb_return_mae = mean_absolute_error(
    y_return_validation,
    lgb_return_validation_pred
)

lgb_return_rmse = np.sqrt(
    mean_squared_error(
        y_return_validation,
        lgb_return_validation_pred
    )
)

lgb_return_r2 = r2_score(
    y_return_validation,
    lgb_return_validation_pred
)

# ------------------------------------------------------------
# DIRECTION ACCURACY
# ------------------------------------------------------------

actual_direction = np.sign(
    y_return_validation.values
)

predicted_direction = np.sign(
    lgb_return_validation_pred
)

lgb_direction_accuracy = np.mean(
    actual_direction == predicted_direction
) * 100

# ------------------------------------------------------------
# RETURN CORRELATION
# ------------------------------------------------------------

lgb_return_correlation = np.corrcoef(
    y_return_validation.values,
    lgb_return_validation_pred
)[0, 1]

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("LIGHTGBM - 1D RETURN REGRESSION")
print("--------------------------------")
print("Best iteration       :", lgb_return_model.best_iteration_)
print("R²                   :", lgb_return_r2)
print("MAE                  :", lgb_return_mae)
print("RMSE                 :", lgb_return_rmse)
print("Direction Accuracy   :", lgb_direction_accuracy, "%")
print("Return Correlation   :", lgb_return_correlation)

LIGHTGBM - 1D RETURN REGRESSION
--------------------------------
Best iteration       : 11
R²                   : -0.0009449094147813142
MAE                  : 0.012621874639896329
RMSE                 : 0.01830694072599342
Direction Accuracy   : 47.636550760861574 %
Return Correlation   : 0.00257458192701261


In [51]:
# ============================================================
# XGBOOST - 1D RETURN REGRESSION
# ============================================================

from xgboost import XGBRegressor
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

xgb_return_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,

    max_depth=5,
    min_child_weight=10,

    subsample=0.8,
    colsample_bytree=0.8,

    gamma=0.05,

    reg_alpha=0.1,
    reg_lambda=2.0,

    objective="reg:squarederror",

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

xgb_return_model.fit(
    X_return_train,
    y_return_train,

    eval_set=[
        (X_return_validation, y_return_validation)
    ],

    verbose=False
)

# ------------------------------------------------------------
# VALIDATION PREDICTION
# ------------------------------------------------------------

xgb_return_validation_pred = (
    xgb_return_model.predict(
        X_return_validation
    )
)

# ------------------------------------------------------------
# REGRESSION METRICS
# ------------------------------------------------------------

xgb_return_mae = mean_absolute_error(
    y_return_validation,
    xgb_return_validation_pred
)

xgb_return_rmse = np.sqrt(
    mean_squared_error(
        y_return_validation,
        xgb_return_validation_pred
    )
)

xgb_return_r2 = r2_score(
    y_return_validation,
    xgb_return_validation_pred
)

# ------------------------------------------------------------
# DIRECTION ACCURACY
# ------------------------------------------------------------

xgb_actual_direction = np.sign(
    y_return_validation.values
)

xgb_predicted_direction = np.sign(
    xgb_return_validation_pred
)

xgb_direction_accuracy = np.mean(
    xgb_actual_direction == xgb_predicted_direction
) * 100

# ------------------------------------------------------------
# RETURN CORRELATION
# ------------------------------------------------------------

xgb_return_correlation = np.corrcoef(
    y_return_validation.values,
    xgb_return_validation_pred
)[0, 1]

# ------------------------------------------------------------
# PREDICTION STATISTICS
# ------------------------------------------------------------

print("XGBOOST - 1D RETURN REGRESSION")
print("-------------------------------")

print("R²                   :", xgb_return_r2)
print("MAE                  :", xgb_return_mae)
print("RMSE                 :", xgb_return_rmse)
print("Direction Accuracy   :", xgb_direction_accuracy, "%")
print("Return Correlation   :", xgb_return_correlation)

print("\nPrediction statistics:")
print("Min prediction       :", xgb_return_validation_pred.min())
print("Mean prediction      :", xgb_return_validation_pred.mean())
print("Max prediction       :", xgb_return_validation_pred.max())

XGBOOST - 1D RETURN REGRESSION
-------------------------------
R²                   : -0.0020963034740353326
MAE                  : 0.012621826754060173
RMSE                 : 0.018317467001911673
Direction Accuracy   : 47.463794751157835 %
Return Correlation   : 0.0009291344525257037

Prediction statistics:
Min prediction       : 4.5678694e-06
Mean prediction      : 0.00055097643
Max prediction       : 0.033181414


In [52]:
# ============================================================
# CROSS-SECTIONAL RANKING TARGET
# ============================================================

# Work from the clean model dataset
df_rank = df_model.copy()

# Rank stocks within each trading day by their ACTUAL next-day return.
# Higher rank = better next-day performance.
df_rank["RETURN_RANK"] = (
    df_rank
    .groupby("Date")["TARGET_RETURN_1D"]
    .rank(
        method="average",
        ascending=False
    )
)

# Convert ranking into a normalized score:
# 1.0 = strongest stock that day
# 0.0 = weakest stock that day
daily_stock_count = (
    df_rank.groupby("Date")["SYMBOL"]
    .transform("count")
)

df_rank["RANK_SCORE"] = (
    1
    - (df_rank["RETURN_RANK"] - 1)
    / (daily_stock_count - 1)
)

print("CROSS-SECTIONAL RANKING DATA")
print("-----------------------------")
print("Rows        :", len(df_rank))
print("Features    :", len(feature_cols))
print("Stocks      :", df_rank["SYMBOL"].nunique())
print("Dates       :", df_rank["Date"].nunique())

print("\nRank score:")
print("Minimum     :", df_rank["RANK_SCORE"].min())
print("Mean        :", df_rank["RANK_SCORE"].mean())
print("Maximum     :", df_rank["RANK_SCORE"].max())

print("\nSYMBOL included:", "SYMBOL" in feature_cols)


CROSS-SECTIONAL RANKING DATA
-----------------------------
Rows        : 264181
Features    : 44
Stocks      : 500
Dates       : 560

Rank score:
Minimum     : 0.0
Mean        : 0.5
Maximum     : 1.0

SYMBOL included: False


In [53]:
# ============================================================
# RANKING MODEL — CHRONOLOGICAL SPLIT
# ============================================================

df_rank = df_rank.sort_values(
    ["Date", "SYMBOL"]
).reset_index(drop=True)

rank_unique_dates = np.sort(
    df_rank["Date"].unique()
)

rank_train_end = rank_unique_dates[
    int(len(rank_unique_dates) * 0.70)
]

rank_validation_end = rank_unique_dates[
    int(len(rank_unique_dates) * 0.80)
]

rank_train_mask = (
    df_rank["Date"] < rank_train_end
)

rank_validation_mask = (
    (df_rank["Date"] >= rank_train_end) &
    (df_rank["Date"] < rank_validation_end)
)

rank_test_mask = (
    df_rank["Date"] >= rank_validation_end
)

X_rank_train = df_rank.loc[
    rank_train_mask, feature_cols
].copy()

y_rank_train = df_rank.loc[
    rank_train_mask, "RANK_SCORE"
].copy()

X_rank_validation = df_rank.loc[
    rank_validation_mask, feature_cols
].copy()

y_rank_validation = df_rank.loc[
    rank_validation_mask, "RANK_SCORE"
].copy()

X_rank_test = df_rank.loc[
    rank_test_mask, feature_cols
].copy()

y_rank_test = df_rank.loc[
    rank_test_mask, "RANK_SCORE"
].copy()

# Ranking group sizes
rank_train_groups = (
    df_rank.loc[rank_train_mask]
    .groupby("Date")
    .size()
    .tolist()
)

rank_validation_groups = (
    df_rank.loc[rank_validation_mask]
    .groupby("Date")
    .size()
    .tolist()
)

rank_test_groups = (
    df_rank.loc[rank_test_mask]
    .groupby("Date")
    .size()
    .tolist()
)

print("RANKING MODEL SPLIT")
print("-------------------")

print("Training rows   :", len(X_rank_train))
print("Validation rows :", len(X_rank_validation))
print("Testing rows    :", len(X_rank_test))

print("Training dates  :", len(rank_train_groups))
print("Validation dates:", len(rank_validation_groups))
print("Testing dates   :", len(rank_test_groups))

print("\nTraining period:")
print(
    df_rank.loc[rank_train_mask, "Date"].min(),
    "to",
    df_rank.loc[rank_train_mask, "Date"].max()
)

print("\nValidation period:")
print(
    df_rank.loc[rank_validation_mask, "Date"].min(),
    "to",
    df_rank.loc[rank_validation_mask, "Date"].max()
)

print("\nTesting period:")
print(
    df_rank.loc[rank_test_mask, "Date"].min(),
    "to",
    df_rank.loc[rank_test_mask, "Date"].max()
)

print("\nAverage stocks per day:")
print(
    "Train      :", np.mean(rank_train_groups),
    "\nValidation :", np.mean(rank_validation_groups),
    "\nTest       :", np.mean(rank_test_groups)
)

print("\nSYMBOL included:", "SYMBOL" in feature_cols)

RANKING MODEL SPLIT
-------------------
Training rows   : 181230
Validation rows : 27206
Testing rows    : 55745
Training dates  : 392
Validation dates: 56
Testing dates   : 112

Training period:
2024-03-13 00:00:00 to 2025-10-13 00:00:00

Validation period:
2025-10-14 00:00:00 to 2026-01-02 00:00:00

Testing period:
2026-01-05 00:00:00 to 2026-06-17 00:00:00

Average stocks per day:
Train      : 462.32142857142856 
Validation : 485.82142857142856 
Test       : 497.7232142857143

SYMBOL included: False


In [55]:
# ============================================================
# LIGHTGBM LAMBDARANK
# Convert continuous rank score -> integer relevance 0-100
# ============================================================

import lightgbm as lgb
import numpy as np

# ------------------------------------------------------------
# Convert ranking score to integer relevance
# ------------------------------------------------------------

y_rank_train_lgb = np.round(
    y_rank_train * 100
).astype(int)

y_rank_validation_lgb = np.round(
    y_rank_validation * 100
).astype(int)

# ------------------------------------------------------------
# LambdaRank model
# ------------------------------------------------------------

rank_model = lgb.LGBMRanker(
    objective="lambdarank",

    n_estimators=1000,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    # Relevance levels 0 -> 100
    label_gain=list(range(101)),

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

rank_model.fit(
    X_rank_train,
    y_rank_train_lgb,

    group=rank_train_groups,

    eval_set=[
        (X_rank_validation, y_rank_validation_lgb)
    ],

    eval_group=[
        rank_validation_groups
    ],

    eval_at=[5, 10, 20],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=False
        ),
        lgb.log_evaluation(0)
    ]
)

print("LIGHTGBM LAMBDARANK")
print("-------------------")
print("Best iteration:", rank_model.best_iteration_)

LIGHTGBM LAMBDARANK
-------------------
Best iteration: 124


In [56]:
# ============================================================
# LAMBDARANK — REAL WORLD TEST
# ============================================================

# Predict ranking score for completely unseen test period
rank_test_pred = rank_model.predict(
    X_rank_test,
    num_iteration=rank_model.best_iteration_
)

# Create evaluation dataframe
rank_eval = df_rank.loc[
    rank_test_mask,
    ["Date", "SYMBOL", "TARGET_RETURN_1D"]
].copy()

rank_eval["PRED_SCORE"] = rank_test_pred

print("RANKING TEST DATA")
print("-----------------")
print("Rows:", len(rank_eval))
print("Dates:", rank_eval["Date"].nunique())
print("Stocks:", rank_eval["SYMBOL"].nunique())


# ============================================================
# DAILY TOP-K PERFORMANCE
# ============================================================

def evaluate_top_k(data, k):

    daily_returns = []
    daily_hit_rates = []

    for date, group in data.groupby("Date"):

        group = group.sort_values(
            "PRED_SCORE",
            ascending=False
        )

        top_k = group.head(k)

        daily_returns.append(
            top_k["TARGET_RETURN_1D"].mean()
        )

        daily_hit_rates.append(
            (top_k["TARGET_RETURN_1D"] > 0).mean()
        )

    return (
        np.mean(daily_returns),
        np.median(daily_returns),
        np.mean(daily_hit_rates)
    )


# ============================================================
# RESULTS
# ============================================================

print("\nTOP STOCK PERFORMANCE")
print("---------------------")

for k in [5, 10, 20, 50]:

    avg_return, median_return, hit_rate = evaluate_top_k(
        rank_eval,
        k
    )

    print(
        f"Top {k:2d} | "
        f"Avg Daily Return: {avg_return * 100:.4f}% | "
        f"Median: {median_return * 100:.4f}% | "
        f"Positive Days: {hit_rate * 100:.2f}%"
    )

RANKING TEST DATA
-----------------
Rows: 55745
Dates: 112
Stocks: 500

TOP STOCK PERFORMANCE
---------------------
Top  5 | Avg Daily Return: 0.2806% | Median: 0.1990% | Positive Days: 48.75%
Top 10 | Avg Daily Return: 0.1121% | Median: 0.0908% | Positive Days: 47.77%
Top 20 | Avg Daily Return: 0.0635% | Median: 0.0913% | Positive Days: 46.83%
Top 50 | Avg Daily Return: 0.0643% | Median: 0.2130% | Positive Days: 47.50%


In [57]:
# ============================================================
# LAMBDARANK — PROPER REAL-WORLD BACKTEST
# ============================================================

# ------------------------------------------------------------
# 1. Equal-weight benchmark: all stocks
# ------------------------------------------------------------

benchmark_daily = (
    rank_eval
    .groupby("Date")["TARGET_RETURN_1D"]
    .mean()
)

# ------------------------------------------------------------
# 2. Top 5 portfolio
# ------------------------------------------------------------

top5_daily = (
    rank_eval
    .sort_values(
        ["Date", "PRED_SCORE"],
        ascending=[True, False]
    )
    .groupby("Date")
    .head(5)
    .groupby("Date")["TARGET_RETURN_1D"]
    .mean()
)

# ------------------------------------------------------------
# 3. Top 10 portfolio
# ------------------------------------------------------------

top10_daily = (
    rank_eval
    .sort_values(
        ["Date", "PRED_SCORE"],
        ascending=[True, False]
    )
    .groupby("Date")
    .head(10)
    .groupby("Date")["TARGET_RETURN_1D"]
    .mean()
)

# ------------------------------------------------------------
# 4. Calculate cumulative returns
# ------------------------------------------------------------

benchmark_equity = (1 + benchmark_daily).cumprod()
top5_equity = (1 + top5_daily).cumprod()
top10_equity = (1 + top10_daily).cumprod()

# ------------------------------------------------------------
# 5. Maximum drawdown
# ------------------------------------------------------------

def max_drawdown(equity):

    peak = equity.cummax()

    drawdown = (
        equity / peak
    ) - 1

    return drawdown.min()


# ------------------------------------------------------------
# 6. Sharpe ratio
# ------------------------------------------------------------

def sharpe_ratio(returns):

    if returns.std() == 0:
        return 0

    return (
        returns.mean() /
        returns.std()
    ) * np.sqrt(252)


# ------------------------------------------------------------
# 7. Print results
# ------------------------------------------------------------

print("REAL-WORLD BACKTEST")
print("===================")

print("\nBENCHMARK — ALL STOCKS")
print("----------------------")

print(
    "Avg Daily Return :",
    benchmark_daily.mean() * 100,
    "%"
)

print(
    "Cumulative Return:",
    (benchmark_equity.iloc[-1] - 1) * 100,
    "%"
)

print(
    "Sharpe Ratio     :",
    sharpe_ratio(benchmark_daily)
)

print(
    "Max Drawdown     :",
    max_drawdown(benchmark_equity) * 100,
    "%"
)


print("\nTOP 5 LAMBDARANK")
print("----------------")

print(
    "Avg Daily Return :",
    top5_daily.mean() * 100,
    "%"
)

print(
    "Cumulative Return:",
    (top5_equity.iloc[-1] - 1) * 100,
    "%"
)

print(
    "Sharpe Ratio     :",
    sharpe_ratio(top5_daily)
)

print(
    "Max Drawdown     :",
    max_drawdown(top5_equity) * 100,
    "%"
)


print("\nTOP 10 LAMBDARANK")
print("-----------------")

print(
    "Avg Daily Return :",
    top10_daily.mean() * 100,
    "%"
)

print(
    "Cumulative Return:",
    (top10_equity.iloc[-1] - 1) * 100,
    "%"
)

print(
    "Sharpe Ratio     :",
    sharpe_ratio(top10_daily)
)

print(
    "Max Drawdown     :",
    max_drawdown(top10_equity) * 100,
    "%"
)


# ------------------------------------------------------------
# 8. Rank correlation
# ------------------------------------------------------------

daily_rank_ic = []

for date, group in rank_eval.groupby("Date"):

    if len(group) > 2:

        correlation = group[
            "PRED_SCORE"
        ].corr(
            group["TARGET_RETURN_1D"],
            method="spearman"
        )

        if not np.isnan(correlation):
            daily_rank_ic.append(correlation)


daily_rank_ic = np.array(daily_rank_ic)

print("\nRANKING QUALITY")
print("---------------")

print(
    "Mean Rank IC   :",
    daily_rank_ic.mean()
)

print(
    "Median Rank IC :",
    np.median(daily_rank_ic)
)

print(
    "Positive IC %  :",
    (daily_rank_ic > 0).mean() * 100,
    "%"
)

REAL-WORLD BACKTEST

BENCHMARK — ALL STOCKS
----------------------
Avg Daily Return : 0.03244149675117295 %
Cumulative Return: 2.680989205467288 %
Sharpe Ratio     : 0.38624193418377145
Max Drawdown     : -14.685751509192368 %

TOP 5 LAMBDARANK
----------------
Avg Daily Return : 0.2805708603551611 %
Cumulative Return: 33.26832558134558 %
Sharpe Ratio     : 2.025749660484653
Max Drawdown     : -16.53644628175277 %

TOP 10 LAMBDARANK
-----------------
Avg Daily Return : 0.11209935512636045 %
Cumulative Return: 11.382486150061212 %
Sharpe Ratio     : 0.9951168614779118
Max Drawdown     : -18.159583683751524 %

RANKING QUALITY
---------------
Mean Rank IC   : 0.009169246611304189
Median Rank IC : 0.005469910142196262
Positive IC %  : 52.25225225225225 %


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


In [58]:
# ============================================================
# LAMBDARANK — ROBUSTNESS & TRADING COST ANALYSIS
# ============================================================

# ------------------------------------------------------------
# TOP 5 PORTFOLIO
# ------------------------------------------------------------

top5_data = (
    rank_eval
    .sort_values(
        ["Date", "PRED_SCORE"],
        ascending=[True, False]
    )
    .groupby("Date")
    .head(5)
    .copy()
)

# ------------------------------------------------------------
# Daily Top-5 returns
# ------------------------------------------------------------

top5_daily_returns = (
    top5_data
    .groupby("Date")["TARGET_RETURN_1D"]
    .mean()
)

# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

benchmark_returns = (
    rank_eval
    .groupby("Date")["TARGET_RETURN_1D"]
    .mean()
)

# ------------------------------------------------------------
# Excess return
# ------------------------------------------------------------

excess_returns = (
    top5_daily_returns -
    benchmark_returns
)

# ------------------------------------------------------------
# Trading cost assumptions
# ------------------------------------------------------------

costs = [0.001, 0.002, 0.003]

print("LAMBDARANK ROBUSTNESS ANALYSIS")
print("==============================")

print("\nRAW TOP-5")
print("---------")

print(
    "Average Daily Return :",
    top5_daily_returns.mean() * 100,
    "%"
)

print(
    "Median Daily Return  :",
    top5_daily_returns.median() * 100,
    "%"
)

print(
    "Positive Days        :",
    (top5_daily_returns > 0).mean() * 100,
    "%"
)

print(
    "Beat Benchmark Days  :",
    (excess_returns > 0).mean() * 100,
    "%"
)

print(
    "Average Excess Return:",
    excess_returns.mean() * 100,
    "%"
)

# ------------------------------------------------------------
# Cost analysis
# ------------------------------------------------------------

print("\nTRADING COST SENSITIVITY")
print("------------------------")

for cost in costs:

    net_returns = (
        top5_daily_returns - cost
    )

    cumulative = (
        (1 + net_returns).prod() - 1
    ) * 100

    sharpe = (
        net_returns.mean() /
        net_returns.std()
    ) * np.sqrt(252)

    print(
        f"Cost {cost * 100:.2f}% | "
        f"Avg Daily: {net_returns.mean() * 100:.4f}% | "
        f"Cumulative: {cumulative:.2f}% | "
        f"Sharpe: {sharpe:.3f}"
    )

# ------------------------------------------------------------
# Best / worst days
# ------------------------------------------------------------

print("\nTOP-5 RETURN DISTRIBUTION")
print("-------------------------")

print(
    "Best Day   :",
    top5_daily_returns.max() * 100,
    "%"
)

print(
    "Worst Day  :",
    top5_daily_returns.min() * 100,
    "%"
)

print(
    "25th Percentile:",
    top5_daily_returns.quantile(0.25) * 100,
    "%"
)

print(
    "75th Percentile:",
    top5_daily_returns.quantile(0.75) * 100,
    "%"
)

# ------------------------------------------------------------
# Monthly performance
# ------------------------------------------------------------

monthly_top5 = (
    top5_daily_returns
    .resample("ME")
    .apply(lambda x: (1 + x).prod() - 1)
)

print("\nMONTHLY TOP-5 PERFORMANCE")
print("-------------------------")

print(
    "Positive Months:",
    (monthly_top5 > 0).mean() * 100,
    "%"
)

print(
    "Best Month:",
    monthly_top5.max() * 100,
    "%"
)

print(
    "Worst Month:",
    monthly_top5.min() * 100,
    "%"
)

print(
    "Average Month:",
    monthly_top5.mean() * 100,
    "%"
)

# ------------------------------------------------------------
# Quarterly-style stability
# ------------------------------------------------------------

quarterly_top5 = (
    top5_daily_returns
    .resample("QE")
    .apply(lambda x: (1 + x).prod() - 1)
)

print("\nQUARTERLY PERFORMANCE")
print("---------------------")

print(
    "Positive Quarters:",
    (quarterly_top5 > 0).mean() * 100,
    "%"
)

print(
    "Best Quarter:",
    quarterly_top5.max() * 100,
    "%"
)

print(
    "Worst Quarter:",
    quarterly_top5.min() * 100,
    "%"
)

LAMBDARANK ROBUSTNESS ANALYSIS

RAW TOP-5
---------
Average Daily Return : 0.2805708603551611 %
Median Daily Return  : 0.19902472471151736 %
Positive Days        : 55.35714285714286 %
Beat Benchmark Days  : 55.35714285714286 %
Average Excess Return: 0.24812936360398816 %

TRADING COST SENSITIVITY
------------------------
Cost 0.10% | Avg Daily: 0.1806% | Cumulative: 19.17% | Sharpe: 1.304
Cost 0.20% | Avg Daily: 0.0806% | Cumulative: 6.56% | Sharpe: 0.582
Cost 0.30% | Avg Daily: -0.0194% | Cumulative: -4.74% | Sharpe: -0.140

TOP-5 RETURN DISTRIBUTION
-------------------------
Best Day   : 7.980781398274636 %
Worst Day  : -5.712435960154568 %
25th Percentile: -0.8274674338494825 %
75th Percentile: 1.4544684596924085 %

MONTHLY TOP-5 PERFORMANCE
-------------------------
Positive Months: 66.66666666666666 %
Best Month: 19.803384805398807 %
Worst Month: -12.137868839360111 %
Average Month: 5.514601568685073 %

QUARTERLY PERFORMANCE
---------------------
Positive Quarters: 50.0 %
Best Qua

In [59]:
# ============================================================
# LAMBDARANK — VALIDATION PERFORMANCE
# ============================================================

rank_validation_pred = rank_model.predict(
    X_rank_validation,
    num_iteration=rank_model.best_iteration_
)

rank_validation_eval = df_rank.loc[
    rank_validation_mask,
    ["Date", "SYMBOL", "TARGET_RETURN_1D"]
].copy()

rank_validation_eval["PRED_SCORE"] = rank_validation_pred


def validation_top_k(data, k):

    top_k_data = (
        data
        .sort_values(
            ["Date", "PRED_SCORE"],
            ascending=[True, False]
        )
        .groupby("Date")
        .head(k)
    )

    daily_returns = (
        top_k_data
        .groupby("Date")["TARGET_RETURN_1D"]
        .mean()
    )

    return daily_returns


print("LAMBDARANK VALIDATION")
print("=====================")

for k in [5, 10, 20, 50]:

    returns = validation_top_k(
        rank_validation_eval,
        k
    )

    print(
        f"Top {k:2d} | "
        f"Avg Daily Return: {returns.mean() * 100:.4f}% | "
        f"Median: {returns.median() * 100:.4f}% | "
        f"Positive Days: {(returns > 0).mean() * 100:.2f}%"
    )

LAMBDARANK VALIDATION
Top  5 | Avg Daily Return: 0.2369% | Median: 0.2314% | Positive Days: 57.14%
Top 10 | Avg Daily Return: 0.0506% | Median: 0.1033% | Positive Days: 53.57%
Top 20 | Avg Daily Return: 0.0560% | Median: 0.0702% | Positive Days: 51.79%
Top 50 | Avg Daily Return: 0.0577% | Median: 0.0154% | Positive Days: 50.00%


In [60]:
# ============================================================
# CROSS-SECTIONAL FEATURES FOR RANKING
# ============================================================

df_rank = df_rank.sort_values(
    ["Date", "SYMBOL"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Cross-sectional percentile rank
# ------------------------------------------------------------

rank_features = [
    "RET_1D",
    "RET_3D",
    "RET_5D",
    "RET_10D",
    "RET_20D",
    "RSI_14",
    "MACD",
    "MACD_HIST",
    "VOLATILITY_5",
    "VOLATILITY_10",
    "VOLATILITY_20",
    "ATR_PERCENT",
    "BB_POSITION",
    "VOLUME_RATIO",
    "PRICE_SMA20",
    "PRICE_SMA50"
]

new_rank_features = []

for col in rank_features:

    new_col = "CS_RANK_" + col

    df_rank[new_col] = (
        df_rank
        .groupby("Date")[col]
        .rank(pct=True)
    )

    new_rank_features.append(new_col)


# ------------------------------------------------------------
# Relative momentum / volatility features
# ------------------------------------------------------------

df_rank["RISK_ADJUSTED_RET_5D"] = (
    df_rank["RET_5D"] /
    (df_rank["VOLATILITY_20"] + 1e-8)
)

df_rank["RISK_ADJUSTED_RET_10D"] = (
    df_rank["RET_10D"] /
    (df_rank["VOLATILITY_20"] + 1e-8)
)

df_rank["MOMENTUM_ACCELERATION"] = (
    df_rank["RET_5D"] -
    (df_rank["RET_10D"] / 2)
)

df_rank["PRICE_TREND_STRENGTH"] = (
    df_rank["PRICE_SMA20"] *
    df_rank["PRICE_SMA50"]
)

df_rank["MOMENTUM_VOL_RATIO"] = (
    df_rank["RET_10D"] /
    (df_rank["VOLATILITY_10"] + 1e-8)
)

extra_rank_features = [
    "RISK_ADJUSTED_RET_5D",
    "RISK_ADJUSTED_RET_10D",
    "MOMENTUM_ACCELERATION",
    "PRICE_TREND_STRENGTH",
    "MOMENTUM_VOL_RATIO"
]

new_rank_features.extend(extra_rank_features)

print("CROSS-SECTIONAL FEATURES CREATED")
print("--------------------------------")
print("New features :", len(new_rank_features))
print(new_rank_features)

print(
    "\nTotal candidate features:",
    len(feature_cols) + len(new_rank_features)
)

print(
    "Missing values:",
    df_rank[
        new_rank_features
    ].isnull().sum().sum()
)

CROSS-SECTIONAL FEATURES CREATED
--------------------------------
New features : 21
['CS_RANK_RET_1D', 'CS_RANK_RET_3D', 'CS_RANK_RET_5D', 'CS_RANK_RET_10D', 'CS_RANK_RET_20D', 'CS_RANK_RSI_14', 'CS_RANK_MACD', 'CS_RANK_MACD_HIST', 'CS_RANK_VOLATILITY_5', 'CS_RANK_VOLATILITY_10', 'CS_RANK_VOLATILITY_20', 'CS_RANK_ATR_PERCENT', 'CS_RANK_BB_POSITION', 'CS_RANK_VOLUME_RATIO', 'CS_RANK_PRICE_SMA20', 'CS_RANK_PRICE_SMA50', 'RISK_ADJUSTED_RET_5D', 'RISK_ADJUSTED_RET_10D', 'MOMENTUM_ACCELERATION', 'PRICE_TREND_STRENGTH', 'MOMENTUM_VOL_RATIO']

Total candidate features: 65
Missing values: 0


In [61]:
# ============================================================
# 65-FEATURE RANKING DATA
# ============================================================

rank_feature_cols_65 = (
    feature_cols +
    new_rank_features
)

print("65-FEATURE RANKING DATA")
print("-----------------------")
print("Features:", len(rank_feature_cols_65))
print("Missing:", df_rank[rank_feature_cols_65].isnull().sum().sum())

65-FEATURE RANKING DATA
-----------------------
Features: 65
Missing: 0


In [62]:
# ============================================================
# 65-FEATURE LAMBDARANK DATA PREPARATION
# ============================================================

# Integer relevance label for LambdaRank
df_rank["RANK_LABEL_65"] = (
    (df_rank["RANK_SCORE"] * 100)
    .round()
    .astype(int)
)

# ------------------------------------------------------------
# Use the SAME chronological periods as the existing model
# ------------------------------------------------------------

rank65_train_mask = (
    df_rank["Date"] <= pd.Timestamp("2025-10-13")
)

rank65_validation_mask = (
    (df_rank["Date"] >= pd.Timestamp("2025-10-14")) &
    (df_rank["Date"] <= pd.Timestamp("2026-01-02"))
)

rank65_test_mask = (
    df_rank["Date"] >= pd.Timestamp("2026-01-05")
)

# ------------------------------------------------------------
# Features
# ------------------------------------------------------------

X_rank65_train = df_rank.loc[
    rank65_train_mask,
    rank_feature_cols_65
].copy()

X_rank65_validation = df_rank.loc[
    rank65_validation_mask,
    rank_feature_cols_65
].copy()

X_rank65_test = df_rank.loc[
    rank65_test_mask,
    rank_feature_cols_65
].copy()

# ------------------------------------------------------------
# Target
# ------------------------------------------------------------

y_rank65_train = df_rank.loc[
    rank65_train_mask,
    "RANK_LABEL_65"
].copy()

y_rank65_validation = df_rank.loc[
    rank65_validation_mask,
    "RANK_LABEL_65"
].copy()

y_rank65_test = df_rank.loc[
    rank65_test_mask,
    "RANK_LABEL_65"
].copy()

# ------------------------------------------------------------
# Group sizes
# One group = one trading day
# ------------------------------------------------------------

rank65_group_train = (
    df_rank.loc[rank65_train_mask]
    .groupby("Date", sort=True)
    .size()
    .tolist()
)

rank65_group_validation = (
    df_rank.loc[rank65_validation_mask]
    .groupby("Date", sort=True)
    .size()
    .tolist()
)

rank65_group_test = (
    df_rank.loc[rank65_test_mask]
    .groupby("Date", sort=True)
    .size()
    .tolist()
)

print("65-FEATURE LAMBDARANK DATA")
print("==========================")

print("Training rows   :", len(X_rank65_train))
print("Validation rows :", len(X_rank65_validation))
print("Testing rows    :", len(X_rank65_test))
print("Features        :", X_rank65_train.shape[1])

print("\nTraining dates   :", len(rank65_group_train))
print("Validation dates :", len(rank65_group_validation))
print("Testing dates   :", len(rank65_group_test))

print("\nTarget range:")
print(
    "Min:",
    y_rank65_train.min(),
    "Max:",
    y_rank65_train.max()
)

print("\nGroup check:")
print(
    "Train:",
    sum(rank65_group_train),
    "Validation:",
    sum(rank65_group_validation),
    "Test:",
    sum(rank65_group_test)
)

65-FEATURE LAMBDARANK DATA
Training rows   : 181230
Validation rows : 27206
Testing rows    : 55745
Features        : 65

Training dates   : 392
Validation dates : 56
Testing dates   : 112

Target range:
Min: 0 Max: 100

Group check:
Train: 181230 Validation: 27206 Test: 55745


In [63]:
# ============================================================
# 65-FEATURE LIGHTGBM LAMBDARANK
# ============================================================

rank65_model = lgb.LGBMRanker(
    objective="lambdarank",

    n_estimators=1000,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    label_gain=list(range(101)),

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

rank65_model.fit(
    X_rank65_train,
    y_rank65_train,

    group=rank65_group_train,

    eval_set=[
        (X_rank65_validation, y_rank65_validation)
    ],

    eval_group=[
        rank65_group_validation
    ],

    eval_at=[5, 10, 20],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=False
        ),
        lgb.log_evaluation(0)
    ]
)

print("65-FEATURE LIGHTGBM LAMBDARANK")
print("------------------------------")
print(
    "Best iteration:",
    rank65_model.best_iteration_
)

65-FEATURE LIGHTGBM LAMBDARANK
------------------------------
Best iteration: 4


In [64]:
# ============================================================
# 65-FEATURE LAMBDARANK — VALIDATION EVALUATION
# ============================================================

rank65_validation_pred = rank65_model.predict(
    X_rank65_validation,
    num_iteration=rank65_model.best_iteration_
)

rank65_validation_eval = df_rank.loc[
    rank65_validation_mask,
    ["Date", "SYMBOL", "TARGET_RETURN_1D"]
].copy()

rank65_validation_eval["PRED_SCORE"] = rank65_validation_pred


def evaluate_rank65_top_k(data, k):

    top_k = (
        data
        .sort_values(
            ["Date", "PRED_SCORE"],
            ascending=[True, False]
        )
        .groupby("Date")
        .head(k)
    )

    daily_returns = (
        top_k
        .groupby("Date")["TARGET_RETURN_1D"]
        .mean()
    )

    return (
        daily_returns.mean(),
        daily_returns.median(),
        (daily_returns > 0).mean() * 100
    )


print("65-FEATURE LAMBDARANK — VALIDATION")
print("===================================")

for k in [5, 10, 20, 50]:

    avg_ret, median_ret, positive_days = (
        evaluate_rank65_top_k(
            rank65_validation_eval,
            k
        )
    )

    print(
        f"Top {k:2d} | "
        f"Avg Daily Return: {avg_ret * 100:.4f}% | "
        f"Median: {median_ret * 100:.4f}% | "
        f"Positive Days: {positive_days:.2f}%"
    )

print("\nCURRENT 44-FEATURE BASELINE")
print("----------------------------")
print("Top 5 Avg Daily Return: +0.2369%")

65-FEATURE LAMBDARANK — VALIDATION
Top  5 | Avg Daily Return: 0.3439% | Median: 0.3547% | Positive Days: 58.93%
Top 10 | Avg Daily Return: 0.2177% | Median: 0.2301% | Positive Days: 58.93%
Top 20 | Avg Daily Return: 0.1652% | Median: 0.1634% | Positive Days: 62.50%
Top 50 | Avg Daily Return: 0.0661% | Median: -0.0151% | Positive Days: 50.00%

CURRENT 44-FEATURE BASELINE
----------------------------
Top 5 Avg Daily Return: +0.2369%


In [65]:
# ============================================================
# LAMBDARANK HYPERPARAMETER EXPERIMENT
# 65 FEATURES
# ============================================================

rank_experiments = {

    "A_current": {
        "num_leaves": 63,
        "min_child_samples": 100,
        "learning_rate": 0.03,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0
    },

    "B_more_leaves": {
        "num_leaves": 127,
        "min_child_samples": 100,
        "learning_rate": 0.03,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0
    },

    "C_regularized": {
        "num_leaves": 31,
        "min_child_samples": 200,
        "learning_rate": 0.02,
        "reg_alpha": 0.5,
        "reg_lambda": 2.0
    },

    "D_deeper": {
        "num_leaves": 63,
        "min_child_samples": 50,
        "learning_rate": 0.02,
        "reg_alpha": 0.05,
        "reg_lambda": 0.5
    }
}


rank_experiment_results = []


for name, params in rank_experiments.items():

    print(f"\nTraining {name}...")

    model = lgb.LGBMRanker(
        objective="lambdarank",

        n_estimators=1000,

        num_leaves=params["num_leaves"],
        min_child_samples=params["min_child_samples"],
        learning_rate=params["learning_rate"],

        max_depth=-1,

        subsample=0.8,
        colsample_bytree=0.8,

        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],

        label_gain=list(range(101)),

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_rank65_train,
        y_rank65_train,

        group=rank65_group_train,

        eval_set=[
            (X_rank65_validation, y_rank65_validation)
        ],

        eval_group=[
            rank65_group_validation
        ],

        eval_at=[5, 10, 20],

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            ),
            lgb.log_evaluation(0)
        ]
    )

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------

    predictions = model.predict(
        X_rank65_validation,
        num_iteration=model.best_iteration_
    )

    evaluation = df_rank.loc[
        rank65_validation_mask,
        ["Date", "TARGET_RETURN_1D"]
    ].copy()

    evaluation["PRED_SCORE"] = predictions

    # --------------------------------------------------------
    # Top 5
    # --------------------------------------------------------

    top5 = (
        evaluation
        .sort_values(
            ["Date", "PRED_SCORE"],
            ascending=[True, False]
        )
        .groupby("Date")
        .head(5)
    )

    top5_daily = (
        top5
        .groupby("Date")["TARGET_RETURN_1D"]
        .mean()
    )

    # --------------------------------------------------------
    # Top 10
    # --------------------------------------------------------

    top10 = (
        evaluation
        .sort_values(
            ["Date", "PRED_SCORE"],
            ascending=[True, False]
        )
        .groupby("Date")
        .head(10)
    )

    top10_daily = (
        top10
        .groupby("Date")["TARGET_RETURN_1D"]
        .mean()
    )

    # --------------------------------------------------------
    # Top 20
    # --------------------------------------------------------

    top20 = (
        evaluation
        .sort_values(
            ["Date", "PRED_SCORE"],
            ascending=[True, False]
        )
        .groupby("Date")
        .head(20)
    )

    top20_daily = (
        top20
        .groupby("Date")["TARGET_RETURN_1D"]
        .mean()
    )

    rank_experiment_results.append({
        "Model": name,
        "Best Iteration": model.best_iteration_,

        "Top5 Avg Return":
            top5_daily.mean(),

        "Top10 Avg Return":
            top10_daily.mean(),

        "Top20 Avg Return":
            top20_daily.mean(),

        "Top5 Positive Days":
            (top5_daily > 0).mean(),

        "Top10 Positive Days":
            (top10_daily > 0).mean(),

        "Top20 Positive Days":
            (top20_daily > 0).mean()
    })


# ============================================================
# RESULTS
# ============================================================

rank_experiment_results_df = pd.DataFrame(
    rank_experiment_results
)

print("\n")
print("LAMBDARANK HYPERPARAMETER RESULTS")
print("=================================")

print(
    rank_experiment_results_df
    .sort_values(
        "Top5 Avg Return",
        ascending=False
    )
    .to_string(index=False)
)

print("\nCURRENT BEST:")
print("65-feature LambdaRank")
print("Validation Top-5: +0.3439%")


Training A_current...

Training B_more_leaves...

Training C_regularized...

Training D_deeper...


LAMBDARANK HYPERPARAMETER RESULTS
        Model  Best Iteration  Top5 Avg Return  Top10 Avg Return  Top20 Avg Return  Top5 Positive Days  Top10 Positive Days  Top20 Positive Days
    A_current               4         0.003439          0.002177          0.001652            0.589286             0.589286             0.625000
B_more_leaves              11         0.001615          0.001377          0.001261            0.553571             0.517857             0.535714
C_regularized               1         0.001541          0.002624          0.002047            0.464286             0.535714             0.535714
     D_deeper               1         0.000397          0.000369          0.000126            0.571429             0.553571             0.553571

CURRENT BEST:
65-feature LambdaRank
Validation Top-5: +0.3439%


In [66]:
# ============================================================
# FINAL 65-FEATURE LAMBDARANK MODEL
# TRAIN + VALIDATION -> FINAL TEST
# ============================================================

# ------------------------------------------------------------
# Combine training + validation data
# ------------------------------------------------------------

X_rank65_full_train = pd.concat(
    [
        X_rank65_train,
        X_rank65_validation
    ],
    axis=0
).reset_index(drop=True)

y_rank65_full_train = pd.concat(
    [
        y_rank65_train,
        y_rank65_validation
    ],
    axis=0
).reset_index(drop=True)


# ------------------------------------------------------------
# Combine ranking groups
# ------------------------------------------------------------

rank65_full_group = (
    rank65_group_train +
    rank65_group_validation
)


print("FINAL LAMBDARANK TRAINING DATA")
print("------------------------------")
print("Rows     :", len(X_rank65_full_train))
print("Features :", X_rank65_full_train.shape[1])
print("Dates    :", len(rank65_full_group))
print("Groups sum:", sum(rank65_full_group))


# ============================================================
# FINAL MODEL
# ============================================================

final_rank_model = lgb.LGBMRanker(

    objective="lambdarank",

    # Selected using validation
    n_estimators=4,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=-1,

    min_child_samples=100,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    label_gain=list(range(101)),

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


final_rank_model.fit(
    X_rank65_full_train,
    y_rank65_full_train,

    group=rank65_full_group
)


print("\nFINAL MODEL TRAINED")
print("-------------------")
print("Features :", X_rank65_full_train.shape[1])
print("Trees    :", 4)

FINAL LAMBDARANK TRAINING DATA
------------------------------
Rows     : 208436
Features : 65
Dates    : 448
Groups sum: 208436

FINAL MODEL TRAINED
-------------------
Features : 65
Trees    : 4


In [67]:
# ============================================================
# FINAL LAMBDARANK — UNSEEN TEST EVALUATION
# ============================================================

# ------------------------------------------------------------
# Predict on completely unseen test period
# ------------------------------------------------------------

final_rank_test_pred = final_rank_model.predict(
    X_rank65_test
)


# ------------------------------------------------------------
# Build test evaluation dataframe
# ------------------------------------------------------------

final_rank_test_eval = df_rank.loc[
    rank65_test_mask,
    [
        "Date",
        "SYMBOL",
        "TARGET_RETURN_1D"
    ]
].copy()

final_rank_test_eval["PRED_SCORE"] = final_rank_test_pred


# ============================================================
# TOP-K PERFORMANCE
# ============================================================

print("FINAL LAMBDARANK — TEST RESULTS")
print("===============================")

top_k_results = {}

for k in [5, 10, 20, 50]:

    top_k = (
        final_rank_test_eval
        .sort_values(
            ["Date", "PRED_SCORE"],
            ascending=[True, False]
        )
        .groupby("Date")
        .head(k)
    )

    daily_returns = (
        top_k
        .groupby("Date")["TARGET_RETURN_1D"]
        .mean()
    )

    avg_return = daily_returns.mean()
    median_return = daily_returns.median()
    positive_days = (daily_returns > 0).mean()

    top_k_results[k] = daily_returns

    print(
        f"Top {k:2d} | "
        f"Avg Daily Return: {avg_return * 100:.4f}% | "
        f"Median: {median_return * 100:.4f}% | "
        f"Positive Days: {positive_days * 100:.2f}%"
    )


# ============================================================
# ALL-STOCK BENCHMARK
# ============================================================

benchmark_daily = (
    final_rank_test_eval
    .groupby("Date")["TARGET_RETURN_1D"]
    .mean()
)

print("\nBENCHMARK — ALL STOCKS")
print("----------------------")

print(
    "Avg Daily Return :",
    benchmark_daily.mean() * 100,
    "%"
)

print(
    "Median Daily Return :",
    benchmark_daily.median() * 100,
    "%"
)

print(
    "Positive Days :",
    (benchmark_daily > 0).mean() * 100,
    "%"
)


# ============================================================
# EXCESS RETURN — TOP 5
# ============================================================

top5_daily = top_k_results[5]

excess_return = (
    top5_daily.values -
    benchmark_daily.loc[top5_daily.index].values
)

print("\nTOP-5 VS BENCHMARK")
print("------------------")

print(
    "Top-5 Avg Return :",
    top5_daily.mean() * 100,
    "%"
)

print(
    "Benchmark Avg Return :",
    benchmark_daily.loc[top5_daily.index].mean() * 100,
    "%"
)

print(
    "Average Excess Return :",
    excess_return.mean() * 100,
    "%"
)

print(
    "Beat Benchmark Days :",
    (excess_return > 0).mean() * 100,
    "%"
)

FINAL LAMBDARANK — TEST RESULTS
Top  5 | Avg Daily Return: 0.3286% | Median: 0.1967% | Positive Days: 55.36%
Top 10 | Avg Daily Return: 0.1749% | Median: 0.1641% | Positive Days: 53.57%
Top 20 | Avg Daily Return: 0.1257% | Median: 0.0629% | Positive Days: 51.79%
Top 50 | Avg Daily Return: 0.1038% | Median: 0.0842% | Positive Days: 55.36%

BENCHMARK — ALL STOCKS
----------------------
Avg Daily Return : 0.03244149675117295 %
Median Daily Return : 0.11191507331792588 %
Positive Days : 55.35714285714286 %

TOP-5 VS BENCHMARK
------------------
Top-5 Avg Return : 0.32860457890125955 %
Benchmark Avg Return : 0.03244149675117295 %
Average Excess Return : 0.2961630821500866 %
Beat Benchmark Days : 61.60714285714286 %


In [68]:
# ============================================================
# FINAL MODEL — REAL-WORLD ROBUSTNESS TEST
# ============================================================

top5_daily = top_k_results[5].copy()

# ------------------------------------------------------------
# Basic statistics
# ------------------------------------------------------------

avg_daily = top5_daily.mean()
median_daily = top5_daily.median()

positive_days = (
    top5_daily > 0
).mean()

best_day = top5_daily.max()
worst_day = top5_daily.min()

# ------------------------------------------------------------
# Cumulative return
# ------------------------------------------------------------

cumulative_curve = (
    (1 + top5_daily)
    .cumprod()
)

cumulative_return = (
    cumulative_curve.iloc[-1] - 1
)

# ------------------------------------------------------------
# Sharpe ratio
# ------------------------------------------------------------

sharpe_ratio = (
    avg_daily /
    top5_daily.std()
) * np.sqrt(252)

# ------------------------------------------------------------
# Maximum drawdown
# ------------------------------------------------------------

running_max = cumulative_curve.cummax()

drawdown = (
    cumulative_curve / running_max
) - 1

max_drawdown = drawdown.min()


print("FINAL LAMBDARANK — ROBUSTNESS")
print("=============================")

print(
    f"Average Daily Return : {avg_daily * 100:.4f}%"
)

print(
    f"Median Daily Return  : {median_daily * 100:.4f}%"
)

print(
    f"Positive Days        : {positive_days * 100:.2f}%"
)

print(
    f"Best Day             : {best_day * 100:.4f}%"
)

print(
    f"Worst Day            : {worst_day * 100:.4f}%"
)

print(
    f"Cumulative Return    : {cumulative_return * 100:.4f}%"
)

print(
    f"Sharpe Ratio         : {sharpe_ratio:.4f}"
)

print(
    f"Maximum Drawdown     : {max_drawdown * 100:.4f}%"
)


# ============================================================
# TRANSACTION COST SENSITIVITY
# ============================================================

print("\nTRANSACTION COST SENSITIVITY")
print("============================")

for cost in [0.001, 0.002, 0.003, 0.005]:

    net_daily = top5_daily - cost

    net_curve = (
        (1 + net_daily)
        .cumprod()
    )

    net_cumulative = (
        net_curve.iloc[-1] - 1
    )

    net_sharpe = (
        net_daily.mean() /
        net_daily.std()
    ) * np.sqrt(252)

    print(
        f"Cost {cost * 100:.2f}% | "
        f"Avg Daily: {net_daily.mean() * 100:.4f}% | "
        f"Cumulative: {net_cumulative * 100:.2f}% | "
        f"Sharpe: {net_sharpe:.3f}"
    )


# ============================================================
# MONTHLY PERFORMANCE
# ============================================================

monthly_returns = (
    (1 + top5_daily)
    .groupby(
        top5_daily.index.to_period("M")
    )
    .prod()
    - 1
)

print("\nMONTHLY PERFORMANCE")
print("===================")

print(
    f"Positive Months : "
    f"{(monthly_returns > 0).mean() * 100:.2f}%"
)

print(
    f"Best Month      : "
    f"{monthly_returns.max() * 100:.2f}%"
)

print(
    f"Worst Month     : "
    f"{monthly_returns.min() * 100:.2f}%"
)

print(
    f"Average Month   : "
    f"{monthly_returns.mean() * 100:.2f}%"
)

FINAL LAMBDARANK — ROBUSTNESS
Average Daily Return : 0.3286%
Median Daily Return  : 0.1967%
Positive Days        : 55.36%
Best Day             : 6.8678%
Worst Day            : -6.0547%
Cumulative Return    : 41.4023%
Sharpe Ratio         : 2.6710
Maximum Drawdown     : -17.2785%

TRANSACTION COST SENSITIVITY
Cost 0.10% | Avg Daily: 0.2286% | Cumulative: 26.45% | Sharpe: 1.858
Cost 0.20% | Avg Daily: 0.1286% | Cumulative: 13.07% | Sharpe: 1.045
Cost 0.30% | Avg Daily: 0.0286% | Cumulative: 1.10% | Sharpe: 0.233
Cost 0.50% | Avg Daily: -0.1714% | Cumulative: -19.21% | Sharpe: -1.393

MONTHLY PERFORMANCE
Positive Months : 66.67%
Best Month      : 21.63%
Worst Month     : -9.59%
Average Month   : 6.48%


In [69]:
# ============================================================
# FINAL TOP-5 PORTFOLIO TURNOVER
# ============================================================

top5_positions = (
    final_rank_test_eval
    .sort_values(
        ["Date", "PRED_SCORE"],
        ascending=[True, False]
    )
    .groupby("Date")
    .head(5)
)

# Create set of Top-5 stocks for each day
daily_top5 = (
    top5_positions
    .groupby("Date")["SYMBOL"]
    .apply(set)
    .sort_index()
)

turnover_values = []

dates = daily_top5.index

for i in range(1, len(dates)):

    previous_stocks = daily_top5.iloc[i - 1]
    current_stocks = daily_top5.iloc[i]

    # Equal-weight portfolio.
    # 1.0 = completely different portfolio
    # 0.0 = exactly the same portfolio
    overlap = len(
        previous_stocks.intersection(current_stocks)
    )

    turnover = 1 - (overlap / 5)

    turnover_values.append(turnover)


turnover_series = pd.Series(
    turnover_values,
    index=dates[1:]
)


print("TOP-5 PORTFOLIO TURNOVER")
print("========================")

print(
    "Average daily turnover :",
    turnover_series.mean() * 100,
    "%"
)

print(
    "Median daily turnover  :",
    turnover_series.median() * 100,
    "%"
)

print(
    "Days with no change    :",
    (turnover_series == 0).mean() * 100,
    "%"
)

print(
    "Days with full change  :",
    (turnover_series == 1).mean() * 100,
    "%"
)

print(
    "Maximum turnover       :",
    turnover_series.max() * 100,
    "%"
)

TOP-5 PORTFOLIO TURNOVER
Average daily turnover : 80.0 %
Median daily turnover  : 80.0 %
Days with no change    : 0.0 %
Days with full change  : 34.234234234234236 %
Maximum turnover       : 100.0 %


In [70]:
# ============================================================
# HOLDING PERIOD ANALYSIS
# FINAL LAMBDARANK TOP-5
# ============================================================

# Get daily Top-5 portfolio
top5_positions = (
    final_rank_test_eval
    .sort_values(
        ["Date", "PRED_SCORE"],
        ascending=[True, False]
    )
    .groupby("Date")
    .head(5)
)

daily_top5 = (
    top5_positions
    .groupby("Date")["SYMBOL"]
    .apply(list)
    .sort_index()
)

print("HOLDING PERIOD ANALYSIS")
print("=======================")

for holding_days in [1, 2, 3, 5]:

    portfolio_returns = []

    dates = daily_top5.index

    for i in range(
        0,
        len(dates) - holding_days + 1,
        holding_days
    ):

        selected_date = dates[i]
        selected_stocks = daily_top5.loc[selected_date]

        period_dates = dates[
            i + 1 :
            i + holding_days + 1
        ]

        if len(period_dates) == 0:
            continue

        period_data = final_rank_test_eval[
            final_rank_test_eval["Date"].isin(period_dates) &
            final_rank_test_eval["SYMBOL"].isin(selected_stocks)
        ]

        if len(period_data) == 0:
            continue

        # Equal-weight average return
        period_return = (
            period_data
            .groupby("Date")["TARGET_RETURN_1D"]
            .mean()
            .sum()
        )

        portfolio_returns.append(period_return)

    portfolio_returns = np.array(portfolio_returns)

    print(
        f"\nHolding {holding_days} day(s)"
    )

    print(
        "Average period return :",
        portfolio_returns.mean() * 100,
        "%"
    )

    print(
        "Positive periods      :",
        (portfolio_returns > 0).mean() * 100,
        "%"
    )

    print(
        "Best period           :",
        portfolio_returns.max() * 100,
        "%"
    )

    print(
        "Worst period          :",
        portfolio_returns.min() * 100,
        "%"
    )

HOLDING PERIOD ANALYSIS

Holding 1 day(s)
Average period return : 0.1885158740516024 %
Positive periods      : 56.75675675675676 %
Best period           : 6.080608528627578 %
Worst period          : -5.692155006248609 %

Holding 2 day(s)
Average period return : 0.09230108586388153 %
Positive periods      : 51.78571428571429 %
Best period           : 6.568326900743091 %
Worst period          : -6.902408107907709 %

Holding 3 day(s)
Average period return : 0.04841935625732689 %
Positive periods      : 43.24324324324324 %
Best period           : 6.331413687696696 %
Worst period          : -5.302496950512727 %

Holding 5 day(s)
Average period return : -0.4277327818690408 %
Positive periods      : 50.0 %
Best period           : 5.847205602632595 %
Worst period          : -8.416559848717295 %


In [71]:
# ============================================================
# RETURN CALIBRATION — PREPARE VALIDATION DATA
# ============================================================

# LambdaRank prediction on validation data
calibration_validation_pred = rank65_model.predict(
    X_rank65_validation,
    num_iteration=rank65_model.best_iteration_
)

# Build calibration dataframe
calibration_validation = df_rank.loc[
    rank65_validation_mask,
    [
        "Date",
        "SYMBOL",
        "TARGET_RETURN_1D"
    ]
].copy()

calibration_validation["RANK_SCORE_MODEL"] = (
    calibration_validation_pred
)

print("RETURN CALIBRATION DATA")
print("-----------------------")
print("Rows      :", len(calibration_validation))
print("Features  :", 1)
print("Target    : TARGET_RETURN_1D")

print("\nRank score statistics:")
print(
    "Minimum :", calibration_validation["RANK_SCORE_MODEL"].min()
)

print(
    "Mean    :", calibration_validation["RANK_SCORE_MODEL"].mean()
)

print(
    "Maximum :", calibration_validation["RANK_SCORE_MODEL"].max()
)

print("\nTarget statistics:")
print(
    "Mean    :",
    calibration_validation["TARGET_RETURN_1D"].mean()
)

print(
    "Std     :",
    calibration_validation["TARGET_RETURN_1D"].std()
)

print(
    "Missing :",
    calibration_validation[
        ["RANK_SCORE_MODEL", "TARGET_RETURN_1D"]
    ].isnull().sum().sum()
)

RETURN CALIBRATION DATA
-----------------------
Rows      : 27206
Features  : 1
Target    : TARGET_RETURN_1D

Rank score statistics:
Minimum : -0.07378729167552664
Mean    : -0.01163790739746893
Maximum : 0.10285363692417776

Target statistics:
Mean    : 0.0002683424373391433
Std     : 0.018298633951234735
Missing : 0


In [72]:
# ============================================================
# RANK SCORE -> ACTUAL RETURN RELATIONSHIP
# ============================================================

# ------------------------------------------------------------
# Correlation
# ------------------------------------------------------------

pearson_corr = (
    calibration_validation[
        "RANK_SCORE_MODEL"
    ].corr(
        calibration_validation["TARGET_RETURN_1D"],
        method="pearson"
    )
)

spearman_corr = (
    calibration_validation[
        "RANK_SCORE_MODEL"
    ].corr(
        calibration_validation["TARGET_RETURN_1D"],
        method="spearman"
    )
)

print("RANK SCORE / RETURN RELATIONSHIP")
print("================================")

print("Pearson correlation  :", pearson_corr)
print("Spearman correlation :", spearman_corr)


# ============================================================
# SCORE BUCKET ANALYSIS
# ============================================================

calibration_validation["SCORE_BUCKET"] = pd.qcut(
    calibration_validation["RANK_SCORE_MODEL"],
    q=10,
    labels=False,
    duplicates="drop"
)

bucket_analysis = (
    calibration_validation
    .groupby("SCORE_BUCKET")
    .agg(
        Mean_Score=("RANK_SCORE_MODEL", "mean"),
        Mean_Return=("TARGET_RETURN_1D", "mean"),
        Median_Return=("TARGET_RETURN_1D", "median"),
        Positive_Return=("TARGET_RETURN_1D", lambda x: (x > 0).mean()),
        Count=("TARGET_RETURN_1D", "size")
    )
    .reset_index()
)

bucket_analysis["Mean_Return_%"] = (
    bucket_analysis["Mean_Return"] * 100
)

bucket_analysis["Median_Return_%"] = (
    bucket_analysis["Median_Return"] * 100
)

bucket_analysis["Positive_Return_%"] = (
    bucket_analysis["Positive_Return"] * 100
)

print("\nDECILE ANALYSIS")
print("===============")

print(
    bucket_analysis[
        [
            "SCORE_BUCKET",
            "Mean_Score",
            "Mean_Return_%",
            "Median_Return_%",
            "Positive_Return_%",
            "Count"
        ]
    ].to_string(index=False)
)

RANK SCORE / RETURN RELATIONSHIP
Pearson correlation  : 0.011330990348498686
Spearman correlation : 0.005196052206144786

DECILE ANALYSIS
 SCORE_BUCKET  Mean_Score  Mean_Return_%  Median_Return_%  Positive_Return_%  Count
            0   -0.031857       0.017681        -0.106034          46.742671   3070
            1   -0.024473       0.037154        -0.050837          48.550429   2449
            2   -0.021784       0.063057        -0.059420          47.861965   2666
            3   -0.018700       0.033945        -0.071882          47.056750   2837
            4   -0.015110      -0.002083        -0.087305          47.126796   3202
            5   -0.013247       0.056259        -0.034715          48.776146   2247
            6   -0.009968      -0.005031        -0.066484          47.477745   2696
            7   -0.005079       0.003534        -0.121810          46.343341   2598
            8    0.003250      -0.017629        -0.149719          45.289079   2802
            9    0.023

In [73]:
# ============================================================
# CANDIDATE RETURN MODEL — TRAINING SCORES
# ============================================================

candidate_train_pred = final_rank_model.predict(
    X_rank65_full_train
)

# The full training dataframe is:
# original training + validation
candidate_train_data = df_rank.loc[
    df_rank["Date"] <= pd.Timestamp("2026-01-02"),
    [
        "Date",
        "SYMBOL",
        "TARGET_RETURN_1D"
    ] + rank_feature_cols_65
].copy()

candidate_train_data["RANK_SCORE"] = candidate_train_pred

print("CANDIDATE TRAINING DATA")
print("-----------------------")
print("Rows:", len(candidate_train_data))
print("Features:", len(rank_feature_cols_65))
print("Missing:", candidate_train_data.isnull().sum().sum())

CANDIDATE TRAINING DATA
-----------------------
Rows: 208436
Features: 65
Missing: 0


In [74]:
# ============================================================
# CANDIDATE RETURN MODEL — STRONG STOCKS
# ============================================================

calibration_candidate_data = calibration_validation.copy()

# ------------------------------------------------------------
# Calculate daily LambdaRank percentile
# ------------------------------------------------------------

calibration_candidate_data["RANK_PERCENTILE"] = (
    calibration_candidate_data
    .groupby("Date")["RANK_SCORE_MODEL"]
    .rank(pct=True)
)

# ------------------------------------------------------------
# Strong candidates = top 10% each day
# ------------------------------------------------------------

calibration_candidate_data["IS_STRONG_CANDIDATE"] = (
    calibration_candidate_data["RANK_PERCENTILE"] >= 0.90
)

candidate_data = calibration_candidate_data[
    calibration_candidate_data["IS_STRONG_CANDIDATE"]
].copy()

print("STRONG CANDIDATE DATA")
print("---------------------")

print(
    "Total validation rows :",
    len(calibration_candidate_data)
)

print(
    "Candidate rows        :",
    len(candidate_data)
)

print(
    "Candidate percentage  :",
    len(candidate_data) /
    len(calibration_candidate_data) * 100,
    "%"
)

print("\nCandidate return statistics:")

print(
    "Mean return   :",
    candidate_data["TARGET_RETURN_1D"].mean() * 100,
    "%"
)

print(
    "Median return :",
    candidate_data["TARGET_RETURN_1D"].median() * 100,
    "%"
)

print(
    "Positive rate :",
    (
        candidate_data["TARGET_RETURN_1D"] > 0
    ).mean() * 100,
    "%"
)

print(
    "Std return    :",
    candidate_data["TARGET_RETURN_1D"].std() * 100,
    "%"
)

STRONG CANDIDATE DATA
---------------------
Total validation rows : 27206
Candidate rows        : 2743
Candidate percentage  : 10.082334779092847 %

Candidate return statistics:
Mean return   : 0.06355543623799365 %
Median return : -0.04958841674633785 %
Positive rate : 48.26831935836675 %
Std return    : 2.0303385355583905 %


In [75]:
# ============================================================
# FINAL RETURN CALIBRATION
# ============================================================

# Use the validation period where LambdaRank predictions
# were genuinely out-of-sample.

calibration_top5 = (
    calibration_validation
    .copy()
)

# Rank stocks within each day
calibration_top5["RANK_POSITION"] = (
    calibration_top5
    .groupby("Date")["RANK_SCORE_MODEL"]
    .rank(
        method="first",
        ascending=False
    )
)

# Keep only the stocks that LambdaRank considers Top 5
calibration_top5 = calibration_top5[
    calibration_top5["RANK_POSITION"] <= 5
].copy()

# ------------------------------------------------------------
# Historical expected return of Top-5 candidates
# ------------------------------------------------------------

expected_return = (
    calibration_top5["TARGET_RETURN_1D"].mean()
)

median_return = (
    calibration_top5["TARGET_RETURN_1D"].median()
)

positive_probability = (
    calibration_top5["TARGET_RETURN_1D"] > 0
).mean()

return_std = (
    calibration_top5["TARGET_RETURN_1D"].std()
)

print("FINAL RETURN CALIBRATION")
print("========================")

print(
    "Top-5 candidate rows :",
    len(calibration_top5)
)

print(
    "Expected return      :",
    expected_return * 100,
    "%"
)

print(
    "Median return        :",
    median_return * 100,
    "%"
)

print(
    "Positive probability :",
    positive_probability * 100,
    "%"
)

print(
    "Return volatility    :",
    return_std * 100,
    "%"
)

FINAL RETURN CALIBRATION
Top-5 candidate rows : 280
Expected return      : 0.34393016790218717 %
Median return        : 0.18062457144450983 %
Positive probability : 55.714285714285715 %
Return volatility    : 2.5632111094635026 %


In [76]:
# ============================================================
# SAVE FINAL STOCK PREDICTION MODEL
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

MODEL_DIR = "stock_prediction_model"

os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Save LambdaRank model
# ------------------------------------------------------------

joblib.dump(
    final_rank_model,
    os.path.join(
        MODEL_DIR,
        "lambdarank_model.pkl"
    )
)


# ------------------------------------------------------------
# 2. Save exact 65 feature list
# ------------------------------------------------------------

with open(
    os.path.join(
        MODEL_DIR,
        "feature_columns.json"
    ),
    "w"
) as f:

    json.dump(
        rank_feature_cols_65,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 3. Save return calibration
# ------------------------------------------------------------

calibration_config = {

    "expected_return": float(
        expected_return
    ),

    "median_return": float(
        median_return
    ),

    "positive_probability": float(
        positive_probability
    ),

    "return_volatility": float(
        return_std
    ),

    "calibration_type": "validation_top_5",

    "prediction_horizon": "1D",

    "top_k": 5
}

with open(
    os.path.join(
        MODEL_DIR,
        "calibration.json"
    ),
    "w"
) as f:

    json.dump(
        calibration_config,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 4. Save model configuration
# ------------------------------------------------------------

model_config = {

    "model_type": "LightGBM LambdaRank",

    "objective": "lambdarank",

    "n_estimators": 4,

    "learning_rate": 0.03,

    "num_leaves": 63,

    "max_depth": -1,

    "min_child_samples": 100,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "reg_alpha": 0.1,

    "reg_lambda": 1.0,

    "random_state": 42,

    "number_of_features": 65,

    "number_of_stocks": 500,

    "prediction_horizon": "next trading day",

    "symbol_used_as_feature": False
}

with open(
    os.path.join(
        MODEL_DIR,
        "model_config.json"
    ),
    "w"
) as f:

    json.dump(
        model_config,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 5. Save model metadata / performance
# ------------------------------------------------------------

performance = {

    "validation": {
        "top_5_avg_daily_return": 0.0034393016790218717,
        "top_5_positive_days": 0.5893
    },

    "final_test": {
        "top_5_avg_daily_return": 0.0032860457890125955,
        "top_5_median_daily_return": 0.001967,
        "top_5_positive_days": 0.5536,
        "benchmark_avg_daily_return": 0.0003244149675117295,
        "excess_return": 0.002961630821500866,
        "beat_benchmark_days": 0.6160714,
        "cumulative_return": 0.414023,
        "sharpe_ratio": 2.6710,
        "maximum_drawdown": -0.172785
    }
}

with open(
    os.path.join(
        MODEL_DIR,
        "model_performance.json"
    ),
    "w"
) as f:

    json.dump(
        performance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 6. Save a README for the backend developer
# ------------------------------------------------------------

readme = """
STOCK PREDICTION MODEL
======================

Model:
LightGBM LambdaRank

Purpose:
Rank stocks by expected next-day relative strength.

Features:
65

SYMBOL:
Not used as an ML feature.

Prediction horizon:
Next trading day.

Model file:
lambdarank_model.pkl

Feature list:
feature_columns.json

Configuration:
model_config.json

Return calibration:
calibration.json

Important:
The LambdaRank model is primarily a ranking model.
The calibration value represents the historical expected
return of validation-period Top-5 candidates.

Production flow:

User selects SYMBOL
        ->
Backend obtains market data
        ->
Calculate the exact 65 features
        ->
LambdaRank model
        ->
Ranking / signal
        ->
Return calibration
        ->
Predicted return
        ->
Predicted price
        ->
Direction / confidence
"""

with open(
    os.path.join(
        MODEL_DIR,
        "README.txt"
    ),
    "w"
) as f:

    f.write(readme)


# ------------------------------------------------------------
# 7. Verify files
# ------------------------------------------------------------

print("\nMODEL SAVED")
print("===========")

for filename in sorted(
    os.listdir(MODEL_DIR)
):

    path = os.path.join(
        MODEL_DIR,
        filename
    )

    print(
        f"{filename:<30} "
        f"{os.path.getsize(path):,} bytes"
    )

print("\nLocation:")
print(os.path.abspath(MODEL_DIR))


MODEL SAVED
README.txt                     830 bytes
calibration.json               278 bytes
feature_columns.json           1,264 bytes
lambdarank_model.pkl           36,786 bytes
model_config.json              459 bytes
model_performance.json         551 bytes

Location:
/Users/aryan/Documents/SEM-5/ML/stock_prediction_model
